# TP1 Transformers

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Etape 1 - Télécharger et inspecter le corpus

In [ ]:
from urllib.request import urlretrieve
import os

url = ("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt")

urlretrieve(url, "input.txt")
print(os.path.abspath("input.txt"))

with open("input.txt", "r", encoding="utf-8") as f:
  text = f.read()

print(f"Longueur du corpus : {len(text)}")
print("------")
print(text[:1000])


/content/input.txt
Longueur du corpus : 1115394
------
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak

## Etape 2 - Encodage des caractères

> Pourquoi trier `chars` ? Que garantit le test `decode(encode(message)) == message` ?

Le test garantit que le décodage retrouve bien le message originel. Le tri de `chars` impose un ordre déterministe des caractères car sinon, créer un set ne garantie par d'ordre.

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join([itos[i] for i in ids])

message = "hii there"
print(encode(message))
print(decode(encode(message)))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


## Etape 3 - fabriquer les prédictions attendues

1. Combien de prédictions produit ce bloc ?

> 8

2. Pourquoi ne pas mélanger train et validation ?

> Parce que la validation sert à mesurer si le modèle généralise à des données qu'il n'a pas vues pendant l'entraînement.

3. Pour $t = 3$, quels sont le contexte et la cible ?

> `context = x[:4]` et `target = y[3]` donc `context = [18,47,56,57]` -> `Firs` et `target = 58` -> `t`

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
  context = x[:t + 1]
  target = y[t]
  print(context, "->", target)

tensor([18]) -> tensor(47)
tensor([18, 47]) -> tensor(56)
tensor([18, 47, 56]) -> tensor(57)
tensor([18, 47, 56, 57]) -> tensor(58)
tensor([18, 47, 56, 57, 58]) -> tensor(1)
tensor([18, 47, 56, 57, 58,  1]) -> tensor(15)
tensor([18, 47, 56, 57, 58,  1, 15]) -> tensor(47)
tensor([18, 47, 56, 57, 58,  1, 15, 47]) -> tensor(58)


## Etape 4 - échantillonner un mini-batch

1. Pourquoi retirer `block_size` à la borne de tirage ?

> Car starts donne l'indice de départ pour le dernier bloc commençant qui serait donc `i = len(source)-block_size`

2. Que changerait $B = 32$ ?

> On aurait des batch avec plus d'échantillons dedans (32 séquences au lieu de 4).

3. Que changerait $T = 64$ ?

> Les séquences individuelles seraient plus longues (64 tokens au lieu de 8).

In [ ]:
batch_size = 4
block_size = 8

def get_batch(split):
  source = train_data if split == "train" else val_data
  starts = torch.randint(
      len(source) - block_size, (batch_size,)
  )

  x = torch.stack([
      source[i : i + block_size] for i in starts
  ])

  y = torch.stack([
      source[i + 1 : i + 1 + block_size] for i in starts
  ])

  return x, y

xb, yb = get_batch("train")
print(xb.shape, yb.shape)

torch.Size([4, 8]) torch.Size([4, 8])


## Etape 5 - Une table produit les logits

In [ ]:
class BigramLanguageModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(
        vocab_size, vocab_size
    )

  def forward(self, idx, targets=None):
    logits = self.token_embedding_table(idx)
    # idx: (B,T) -> logits: (B, T, C)
    # B = batch size
    # T = block size
    # C = vocab size
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)
    return logits, loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      logits, loss = self(idx)
      logits = logits[:, -1, :]

      probs = F.softmax(logits, dim=-1)

      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim=1)

    return idx



Pourquoi $C \times C$ ici ? Où retrouvera-t-on $d_{emb}$ et la couche `lm_head` dans le TP4 ?

> La dimension est $C \times C$ car on veut des embeddings qui ont une dimension par token du vocabulaire ($C$ lignes) et la matrice d'embeddings contient tous les embeddings des tokens donc $C$ colonnes.

## Etape 6 et 7 - générer, puis apprendre

In [ ]:
model = BigramLanguageModel(vocab_size)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)


for step in range(100):
  xb, yb = get_batch("train")
  logits, loss = model(xb, yb)

  # nettoyer les gradients à 0
  optimizer.zero_grad(set_to_none=True)
  loss.backward() # backpropag
  optimizer.step()

  if step % 50 == 0:
    print(loss.item())

4.536348342895508
4.195359230041504


In [ ]:
def gen_text_attempt():
  # Génération de texte
  context = torch.zeros((1, 1), dtype=torch.long)

  generated = model.generate(
      context,
      max_new_tokens=500
  )

  print(decode(generated[0].tolist()))

gen_text_attempt()


VHae$.GnTPT3-nOW$wstx l,?Cud:sarXXuh$&ouVhbTPh'LPS
B;u&IlU YcJDvVF&Bj RWz twsMDjYWSC,Oz e
$KJSbLYcZBz'HjCcapSa&nr?DNTOZkql.'RPPhFwRdP
m urtTZrhvnNPymVz mR:JrVOP$gPyEct&mRyHVb!T;uJ;H!D
EwgcNUcpgC;c$tVNONT3apFhiJ;lVzjUU3lKX.,ZsmL&yfq VOaLl
'$?!3aGwplm.Tl.gPz'eh$b&QJ$itKdGhv,fb&cCSrkEIccheT$j-DSuEqxJDXwu.h gbBPZf.h-iBlAKou.tHz!b:VtcsRCA3v;lShf
aFQCSa!Wg&WKHef-
t! ua$!L t3YHA3Hz,Zsq:aP:ogf-d$vMU:T3 lEKEYIAWx-x':Uu'd$;JWiL?Pk.TnMXVq: $r;fBY
HjNDvHbC-q:oDoLMwkkI
m'wvVVoLzMELdB;lbZoMakkC?:abYIcaLl;uqTo


In [ ]:
# continuer l'entraînement
for step in range(900):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
      print(loss.item())

print("=== APRÈS 1000 ÉTAPES ===")
gen_text_attempt()

4.621373176574707
4.3172712326049805
4.179834365844727
4.797718048095703
4.493173599243164
4.58076810836792
4.1505584716796875
4.253966808319092
4.6026082038879395
4.79978609085083
4.2761077880859375
4.416982173919678
4.391849994659424
4.343161582946777
4.103747844696045
4.503380298614502
4.530852317810059
3.8653481006622314
=== APRÈS 1000 ÉTAPES ===

D-ZcNuNPQ3dRRWP&xpwelsdmW,Yr';u.:AbBG$ppasqYmRCs
URYh3g$l.&c!!!HO iSSMVfRWb?!Vf;dmkg$tM?oDPyfjdKBnt;&WoFrm'
lu
H3Qzl.NNQorI;W
hil DnUpfmD
U$-Tid!HF.&eYcXXRmWKDgz f.Fhi.j$jdilAvNQ
k.Jns
NPndKtwqRpW$$ih:TG$Y?IL
FT3a$VD
rtz nC FbIFpHugVO Z&xNDxi,?Rvyfx-rgzESieas,YInl FRu;'.wWWwHaPU iTXxso.G.p'3auVIi&hWkIT-dG$!Y?DNWbBjcZv-3iotwrkCAFN.TPVE&Wvp'YEQPOjEtIAsRClDFwarjMOQpInTAd!H:f.'Y;nTGw!3jJm WSjpKMixW$ndIJQh'IY tPystK- h;nSustTTNq3Pqt!
IUug-EtUScr'YuxZBH,yo:x- ilKNDFkhEqARw;&aeLnoo$TBYCcr'YG.JlSl tWLh.


Pourquoi ne garde-t-on que le dernier pas de temps pendant la génération ? Pourquoi aplatit-on $B \times T$ pendant la loss ?

Comparez la loss et la structure du texte avant/après 100 étapes, puis avec 1 000 étapes.

> Car pour générer le prochain caractère, on ne s'intéresse qu'à la prédiction correspondant au dernier caractère.

> On applatit $B \times T$ pendant la loss car avant, on a des logits de forme `(B,T,C)` et on veut prédire des classes en renvoyant un output de forme `(B,T)`  -> B séquences $\times$ 8 positions = nombre de prédictions. Chaque position est une prédiction indépendante du point de vue de `cross_entropy`.

> Avec plus d'étapes d'entraînement, la loss diminue. Néanmoins, pour l'instant, le texte générer ressemble toujours à du bruit incohérent. Mais en augmentant les epochs, on peut commencer à avoir des certains patterns de lettres qui ressemblent à du vrai texte anglais.

# TP2

## Etape 1 : agréger par produit matriciel

In [ ]:
torch.manual_seed(42)

a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)

b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print(a)
print(b)
print(c)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


## Etape 2 : moyenner uniquement par le passé

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))

for b in range(B):
  for t in range(T):
    xprev = x[b, :t+1] # forme (t+1, C)
    xbow[b, t] = torch.mean(xprev, dim=0)

Formes de `xprev` : `(1, C)` pour `t=0`, `(4, C)` pour `t=3`, `(8, C)` pour `t=7`. On moyenne sur `dim=0` car c'est l'axe temporel de `xprev` (les tokens), pas l'axe des features.

## Etapes 3-4 : vectoriser

In [ ]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x

print(torch.allclose(xbow, xbow2, atol=1e-6))

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x

print(torch.allclose(xbow, xbow3, atol=1e-6))

True
True


`wei @ x` fonctionne car PyTorch **broadcast** la matrice `(T,T)` sur la dimension batch de `x` `(B,T,C)` : chaque batch reçoit indépendamment la même opération `(T,T) @ (T,C) -> (T,C)`. Softmax appliqué à `-inf` donne exactement `0`. C'est ce qui matérialise le masque causal (interdiction de voir le futur) tout en gardant une somme de ligne égale à 1.


## Etape 5 : calculer des poids d'attention

In [ ]:
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
head_size = 16

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x) # (B,T,head_size)
q = query(x) # (B,T,head_size)

# attention scores
wei = q @ k.transpose(-2, -1) # (B,T,T)
wei = wei * head_size**(-0.5) # mise a l’echelle en divisant par sqrt(head_size)
wei = wei.masked_fill(tril[:T, :T] == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v # (B,T,head_size)

 La dernière ligne de `wei[0]` correspond au dernier token, celui qui a accès à tout le contexte passé. La sortie mélange `v` (et non `k`) car `k`/`q` ne servent qu'à *calculer combien* chaque token compte ; `v` porte l'information réellement transmise.

## Etape 6 : stabiliser Softmax

In [ ]:
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

raw = q @ k.transpose(-2, -1)
scaled = raw * head_size**-0.5

print(raw.var(), scaled.var())

z = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print(torch.softmax(z, dim=-1))
print(torch.softmax(z * 8, dim=-1))

tensor(19.8478) tensor(1.2405)
tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


In [ ]:
print(f"Somme des lignes de wei: {wei[0].sum(dim=-1)}")

# Vérification de l'absence de poids vers le futur (éléments masqués à -inf avant softmax)
# Les éléments masqués doivent être proches de zéro après softmax
future_mask = (tril[:T, :T] == 0)
print(f"Vérification des poids futurs (max abs value): {wei[0][future_mask].abs().max()}")

print(f"Forme finale de 'out': {out.shape}")
print(f"Forme attendue de 'out': ({B}, {T}, {head_size})")


Somme des lignes de wei: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)
Vérification des poids futurs (max abs value): 0.0
Forme finale de 'out': torch.Size([4, 8, 16])
Forme attendue de 'out': (4, 8, 16)


#### Vérification de `wei`

1.  **Somme des lignes de `wei` :**
    Chaque ligne de `wei` doit sommer à 1, car elle représente une distribution de probabilité sur les tokens précédents et le token actuel. Cela est garanti par l'application de `F.softmax(wei, dim=-1)`.

2.  **Absence de poids vers le futur :**
    Le masque causal `tril[:T, :T] == 0` remplit les éléments de `wei` correspondant aux tokens futurs avec `-inf` avant l'application du softmax. Après le softmax, ces valeurs deviennent 0, garantissant que le modèle ne peut pas 'voir' les tokens futurs lors du calcul de l'attention pour un token donné.

3.  **Forme finale de l'output :**
    L'opération `out = wei @ v` résulte en une forme `(B, T, head_size)`. Pour chaque séquence dans le batch (`B`), et pour chaque position temporelle (`T`), on obtient un vecteur de taille `head_size` qui est une combinaison pondérée des `v`ecteurs des tokens précédents (et du token actuel) selon les poids d'attention `wei`.

#### Rôle distinct de Q, K et V

-   **Query (Q) :** La requête est ce que le token *recherche* lorsqu'il essaie d'établir des relations avec d'autres tokens. Pour chaque token, `q` est créé à partir de son `x` en utilisant une transformation linéaire (`query`).

-   **Key (K) :** La clé est ce que le token *offre* aux autres tokens pour qu'ils le trouvent pertinent. Pour chaque token, `k` est créé à partir de son `x` en utilisant une transformation linéaire (`key`). Les `q` d'un token sont comparés aux `k` de tous les autres tokens (passés et présents) pour déterminer les poids d'attention.

-   **Value (V) :** La valeur est l'information que le token *transmet* une fois qu'il a été identifié comme pertinent par un autre token. Pour chaque token, `v` est créé à partir de son `x` en utilisant une transformation linéaire (`value`). Ce sont les `v`ecteurs qui sont agrégés (multipliés par les poids d'attention) pour former la sortie `out`.

# TP3 LayerNorm

## Etape 1: coder LayerNorm

In [ ]:
class LayerNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
    def __call__(self, x):
        xmean = x.mean(dim=-1, keepdim=True)
        xvar = x.var(dim=-1, keepdim=True, unbiased=False)
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        return self.gamma * xhat + self.beta


1. Sur quel axe normaliser ? A quoi servent $\epsilon$, $\gamma$, $\beta$ ?
> On normalise sur les features d'un même token donc `dim=-1`(pas sur le batch comme BatchNorm). $\epsilon$ est pris très petit juste pour éviter une potentielle division par 0, tandis que $\gamma$ et $\beta$ sont des paramètres appris par le réseau servant d'échelle/offset si la normalisation stricte (moyenne 0, variance 1) est trop contraignante.

## Etape 2: choisir l'axe de normalisation

In [ ]:
torch.manual_seed(1337)
module = LayerNorm1d(100)

x = torch.randn(32, 100)
x = module(x)

print(x[:, 0].mean(), x[:, 0].std())
print(x[0, :].mean(), x[0, :].std())

torch.allclose(module(x), nn.LayerNorm(100)(x), atol=1e-6)

tensor(0.1476) tensor(0.8847)
tensor(-2.2650e-08) tensor(1.0050)


True

`x[0, :]` devrait donner environ mean = 0, std = 1.

## Etape 3 : construire le réseau feed-forward

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
      super().__init__()
      self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd),
      )

    def forward(self, x):
      return self.net(x)

ffwd = FeedForward(100)
delta = ffwd(torch.randn(32, 100))
print(delta.shape)

torch.Size([32, 100])


Environ la moitié des activations sont positives après ReLU.
On revient à $C$ (`n_embd`) pour que la dimension soit compatible avec le reste du modèle. On avait projetté temporairement sur un espace plus grand pour donner au réseau "plus de place" pour combiner les features non linéairement.

## Etape 4-5 : ajouter le résiduel

In [ ]:
n_embd = 64
layer_norm = nn.LayerNorm(n_embd)
ffwd = FeedForward(n_embd)

x = torch.randn(B, T, n_embd)
delta = ffwd(layer_norm(x))
x_out = x + delta

print(x.shape)
print(delta.shape)
print(x_out.shape)

torch.Size([4, 8, 64])
torch.Size([4, 8, 64])
torch.Size([4, 8, 64])


LayerNorm conserve la dernière dimension (les features donc $C$). Le MLP ne mélange pas les tokens, il s'applique indépendamment à chaque position. La connexion résiduelle (`x + delta`) permet au gradient à être propagé directement à travers les couches profondes sans s'atténuer et permet à chaque bloc d'apprendre une correction plutôt qu'une transformation complète.

# TP4

## Etape 0 : préparer le notebook

In [ ]:
batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3

device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

assert n_embd % n_head == 0
print(device)

cuda


$d_{h} = n_{embd}/n_{head} = 64/4 = 16$.
Un batch contient `batch_size` $\times$ `block_size` = $16*32 = 512$ tokens.
Sur CPU, on réduirait plutôt `n_iters`(les epochs) ou `n_layer` (impacte le nombre de paramètres et de calculs) : c'est ça qui influence le plus le temps d'entraînement.

## Etape 1 : coder une tête

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2,-1) * k.size(-1)**-0.5
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

On utilise `k.size(-1)` plutôt que $C$ pour le facteur d'échelle car ce qui compte pour la stabilisation de la variance est la dimension de la tête, pas la dimension d'entrée.
On tronque avec `tril[:T,:T]` car le buffer est de taille (`block_size`, `block_size`) mais la séquence réellement traitée peut être plus courte (T <= `block_size`) notamment lors de la génération où T grandit progressivement.

## Etape 2 : combiner plusieurs têtes

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

On concatène sur la dernière dimension car c'est celle des features (head_size). Chaque tête produit un morceau du vecteur final.

Si `n_embd` n'était pas divisible par `n_head`, la concaténation des `n_head` sorties de taille `head_size` ne redonneraient pas exactement `n_embd`.

## Etape 3 : assembler un bloc

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

Ordre : LayerNorm > sous couche > addition résiduelle répétée deux fois par bloc. Communication = attention (échange de l'information entre chaque tokens), calcul = MLP (chaque token traite individuellement l'information reçue).

## Etape 4-5-6 : définir les couches du modèle & feed-forward + loss + générer

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) # (B,T,n_embd)
        pos = torch.arange(T, device=idx.device)
        pos_emb = self.position_embedding_table(pos) # (T,n_embd)

        x = tok_emb + pos_emb # (B,T,n_embd)
        x = self.blocks(x) # (B,T,n_embd)
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            logits = logits.reshape(B*T, vocab_size)
            targets = targets.reshape(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss


    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)

            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

* `token_embedding_table` : `(vocab_size, n_embd)`
* `position_embedding_table` : `(block_size, n_embd)`
* `lm_head` : `(n_embd, vocab_size)`

`lm_head` n'est plus confondu avec l'embedding car le modèle a maintenant besoin d'un espace de représentation interne (`n_embd`) séparé de l'espace de sortie (`vocab_size`).

L'addition des embeddings est possible car le broadcasting fait (B,T,n_embd) + (T, n_embd) : PyTorch réplique implicitement pos_emb sur la dimension batch. Chaque token reçoit une représentation qui encode à la fois son identité et sa position dans la séquence.

on n'applatit pas les logits lors de la génération car on a besoin de la structure (B,T,C) pour sélectionner spécifiquement le dernier pas de temps par la suite.

`@torch.no_grad()` : désactive le calcul du graph de gradients qui est inutile en génération, donc on économise de la mémoire (et du temps de génération).
On échantillonne avec multinomial plutôt que argmax pour préserver la diversité dans la génération (argmax est trop déterministe car sélection du token le plus probable à chaque fois).

## Etape 7-8 : évaluer puis entraîner

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            X, Y = X.to(device), Y.to(device)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

model = BigramLanguageModel().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
for iteration in range(max_iters):
    if iteration % eval_interval == 0:
        print(estimate_loss())
    xb, yb = get_batch("train")
    xb, yb = xb.to(device), yb.to(device)
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

{'train': tensor(4.4527), 'val': tensor(4.4423)}
{'train': tensor(2.6592), 'val': tensor(2.6635)}
{'train': tensor(2.4941), 'val': tensor(2.5040)}
{'train': tensor(2.4140), 'val': tensor(2.4287)}
{'train': tensor(2.3386), 'val': tensor(2.3416)}
{'train': tensor(2.2735), 'val': tensor(2.2833)}
{'train': tensor(2.2154), 'val': tensor(2.2223)}
{'train': tensor(2.1706), 'val': tensor(2.1942)}
{'train': tensor(2.1167), 'val': tensor(2.1621)}
{'train': tensor(2.0843), 'val': tensor(2.1270)}
{'train': tensor(2.0449), 'val': tensor(2.0914)}
{'train': tensor(2.0291), 'val': tensor(2.0792)}
{'train': tensor(2.0125), 'val': tensor(2.0753)}
{'train': tensor(1.9642), 'val': tensor(2.0503)}
{'train': tensor(1.9473), 'val': tensor(2.0346)}
{'train': tensor(1.9273), 'val': tensor(2.0263)}
{'train': tensor(1.9134), 'val': tensor(2.0072)}
{'train': tensor(1.8864), 'val': tensor(1.9792)}
{'train': tensor(1.8807), 'val': tensor(1.9890)}
{'train': tensor(1.8613), 'val': tensor(1.9707)}
{'train': tensor(1.8

`model.eval()` désactive le dropout pendant l'évaluation car on veut une mesure stable, pas bruitée.
`model_train()` : réactive le dropout pour la suite de l'entraînement.

Si la loss train baisse mais que la loss validation remonte c'est un signe de surapprentissage (le modèle apprend des patterns vus dans le set de train mais qui ne se généralisent pas bien).

## Etape 9 : entraîner, générer et expérimenter

In [ ]:
model = BigramLanguageModel().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6)  # nb de paramètres (en millions)

# training loop
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iteration in range(max_iters):
    if iteration % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iteration}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device) # Transfert des tenseurs sur le même device
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# génération
context = torch.zeros((1, 1), dtype=torch.long, device=device)
result = model.generate(context, max_new_tokens=2000)
print(decode(result[0].tolist()))

0.209729
step 0: train loss 4.3139, val loss 4.3125
step 100: train loss 2.6466, val loss 2.6567
step 200: train loss 2.5098, val loss 2.5090
step 300: train loss 2.4166, val loss 2.4145
step 400: train loss 2.3405, val loss 2.3419
step 500: train loss 2.2758, val loss 2.2887
step 600: train loss 2.2352, val loss 2.2518
step 700: train loss 2.1979, val loss 2.2067
step 800: train loss 2.1389, val loss 2.1746
step 900: train loss 2.1208, val loss 2.1481
step 1000: train loss 2.0826, val loss 2.1279
step 1100: train loss 2.0374, val loss 2.0843
step 1200: train loss 2.0147, val loss 2.0731
step 1300: train loss 1.9849, val loss 2.0602
step 1400: train loss 1.9646, val loss 2.0535
step 1500: train loss 1.9496, val loss 2.0285
step 1600: train loss 1.9273, val loss 2.0165
step 1700: train loss 1.9179, val loss 2.0095
step 1800: train loss 1.8945, val loss 2.0010
step 1900: train loss 1.8759, val loss 1.9800
step 2000: train loss 1.8707, val loss 1.9782
step 2100: train loss 1.8490, val los

Bilan:

| Classe | Notion | Rôle |
|---|---|---|
| Head / MultiHeadAttention | Self-attention | Contexte : crée la communication entre positions |
| FeedForward | MLP | Calcul : transforme chaque token indépendamment |
| LayerNorm | Normalisation | Stabilise, ne mélange rien |
| Block | Bloc Transformer | Combine communication + calcul avec résiduels |
| token_embedding_table / position_embedding_table | Embeddings | Donnent le sens initial (identité + position) |
| lm_head | Unembedding | Produit les logits finaux sur le vocabulaire |
| F.cross_entropy (dans forward) | Loss | Agit sur l'ensemble du modèle : le gradient qui en découle traverse lm_head, tous les blocs, jusqu'aux embeddings. C'est ce qui met à jour absolument tous les paramètres à chaque pas d'optimisation |